# Fink/LSST — Light Curves by Tag × Deep Drilling Field

This notebook retrieves photometric light curves (detections + forced photometry)  
for objects found via the **Fink LSST classification tags** (`/api/v1/tags` endpoint),  
**restricted to the LSST Deep Drilling Fields** by spatial cross-matching.

## Strategy

1. **Fetch latest alerts per tag** via `GET /api/v1/tags?tag=<tag>&n=N`  
   → retrieve RA/Dec of every alert alongside the standard metadata columns
2. **Spatial filtering per DDF**: for each (tag, DDF) pair, keep only objects  
   whose coordinates fall within `CONE_RADIUS` arcsec of the DDF centre
3. **Deduplicate** by `diaObjectId`, keep objects with `nDiaSources >= NP_MIN`
4. **Download full light curves** via `api/v1/sources` + `api/v1/fp`
5. **Group key = tag** — the Fink tag replaces the `classify_object()` scheme of NB01.  
   The **`field`** (DDF name) is preserved as a separate metadata attribute.
6. **Compute flatness** σ/⟨f⟩ per tag and rank for downstream analysis

## Key API endpoints

```
GET  https://api.lsst.fink-portal.org/api/v1/tags    → alerts filtered by tag
POST https://api.lsst.fink-portal.org/api/v1/sources → full diaSources for one object
POST https://api.lsst.fink-portal.org/api/v1/fp      → forced photometry
GET  https://api.lsst.fink-portal.org/api/v1/tags    → list available tags (no params)
GET  https://api.lsst.fink-portal.org/api/v1/blocks  → list available blocks
```

## Spatial filtering approach

The `/api/v1/tags` endpoint does **not** support spatial (RA/Dec/radius) filtering  
directly. We therefore:
1. Fetch all tag alerts (up to `N_PER_TAG`) globally — includes `r:ra`, `r:dec`
2. Filter client-side using a great-circle distance cut ≤ `CONE_RADIUS` arcsec  
   for each DDF centre

This approach is efficient because we fetch each tag **once** and then apply the  
spatial filter in memory — no extra API calls per DDF.

## Difference from notebook 01

| Aspect | NB 01 (blocks/groups) | NB 07 (tags × DDFs) |
|--------|----------------------|---------------------|
| Object selection | Cone search POST `/conesearch` per DDF | GET `/tags` global then spatial filter per DDF |
| Classification | Custom `classify_object()` on `f:xm_*` | Fink tag = group directly |
| `field` attribute | DDF name from cone search loop | DDF name from spatial filter loop |
| Output dir | `data_FINK_BLOCK_LC_01` | `data_FINK_BLOCK_LC_07` |


## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import io
import time
import collections
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── API base URL ──────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── User parameters ───────────────────────────────────────────────────────────
NP_MIN = 200  # Minimum nDiaSources per object
NC = 50  # Number of light curves to plot per tag
N_PER_TAG = 50000  # Maximum alerts to fetch per tag (global, before spatial filter)
# CONE_RADIUS = 1800.0  # Spatial filter radius in arcsec (0.5 deg — same as NB01)
CONE_RADIUS = 3600.0  # Spatial filter radius in arcsec (1.0 deg — not same as NB01)
BANDS = list("ugrizy")

# ── LSST Deep Drilling Fields (RA/Dec J2000, degrees) ────────────────────────
# Same fields as notebook 01.  Used for spatial filtering after tag fetch.
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Available Fink LSST tags ─────────────────────────────────────────────────
FINK_TAGS = {
    "extragalactic_lt20mag_candidate": "Rising, bright (mag < 20), extragalactic candidates",
    "extragalactic_new_candidate": "New (< 48 h first detection) and potentially extragalactic",
    "hostless_candidate": "Hostless alerts according to ELEPHANT (arXiv:2404.18165)",
    "in_tns": "Alerts with a known counterpart in TNS (AT or confirmed)",
    "sn_near_galaxy_candidate": "Alerts matching a galaxy catalog and consistent with SNe",
}

# Tags to query — by default, all tags. Restrict here if needed.
TAGS_TO_QUERY = list(FINK_TAGS.keys())
# TAGS_TO_QUERY = ["extragalactic_new_candidate", "in_tns"]  # example

# ── Columns requested from /api/v1/tags ──────────────────────────────────────
# r:<col> = diaSource table field (LSST DPDD — 'r:' is the TABLE prefix, not r-band)
# f:<col> = Fink-computed field
CATALOG_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:band",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
        "r:extendedness",
        "r:nDiaSources",
        "r:visit",
        "r:detector",
        "r:x",
        "r:y",
        "f:clf_snnSnVsOthers_score",
        "f:clf_earlySNIa_score",
        "f:clf_cats_class",
        "f:clf_cats_score",
        "f:xm_simbad_otype",
        "f:xm_tns_type",
        "f:xm_tns_fullname",
        "f:xm_legacydr8_zphot",
        "f:xm_legacydr8_pstar",
        "f:xm_mangrove_lum_dist",
        "f:main_label_classifier",
        "f:main_label_crossmatch",
    ]
)

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_07"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")
print(f"Tags to query : {TAGS_TO_QUERY}")
print(f"DDFs          : {list(DEEP_FIELDS.keys())}")

## 2. API wrappers

In [ ]:
def _post_json(url, payload, timeout=90):
    """POST a JSON payload and return the parsed response, raising on HTTP errors."""
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_tag_alerts(tag: str, n: int = N_PER_TAG, columns: str = CATALOG_COLUMNS) -> pd.DataFrame:
    """
    Fetch latest alerts for a given Fink LSST tag (globally, no spatial filter).

    Uses GET /api/v1/tags?tag=<tag>&n=<n>&columns=<cols>&output-format=json
    The spatial filtering (per DDF) is applied client-side afterwards.
    """
    params = {
        "tag": tag,
        "n": n,
        "columns": columns,
        "output-format": "json",
    }
    try:
        resp = requests.get(f"{FINK_API}/api/v1/tags", params=params, timeout=120)
        if resp.status_code in (404, 405):
            print(f"fetch_tag_alerts: endpoint not available (HTTP {resp.status_code})")
            return pd.DataFrame()
        resp.raise_for_status()
        if not resp.text.strip():
            return pd.DataFrame()
        df = pd.read_json(io.BytesIO(resp.content))
        return df
    except Exception as exc:
        print(f"fetch_tag_alerts(tag={tag!r}) error: {exc}")
        return pd.DataFrame()


def list_available_tags() -> pd.DataFrame:
    """List all available Fink LSST tags from /api/v1/tags (no tag parameter)."""
    url = f"{FINK_API}/api/v1/tags"
    try:
        r = requests.get(url, timeout=30)
        if r.status_code in (404, 405):
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            return pd.DataFrame()
        if isinstance(data, list):
            return pd.DataFrame(data) if isinstance(data[0], dict) else pd.DataFrame({"name": data})
        return pd.DataFrame([data]) if isinstance(data, dict) else pd.DataFrame({"raw": [data]})
    except Exception as e:
        print(f"list_available_tags error: {e}")
        return pd.DataFrame()


def list_available_blocks() -> pd.DataFrame:
    """List available Fink blocks from /api/v1/blocks (informational only)."""
    url = f"{FINK_API}/api/v1/blocks"
    try:
        r = requests.get(url, timeout=30)
        if r.status_code in (404, 405):
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            return pd.DataFrame()
        if isinstance(data, list):
            return pd.DataFrame(data) if isinstance(data[0], dict) else pd.DataFrame({"name": data})
        return pd.DataFrame([data]) if isinstance(data, dict) else pd.DataFrame({"raw": [data]})
    except Exception as e:
        print(f"list_available_blocks error: {e}")
        return pd.DataFrame()


def fetch_sources(diaObjectId, columns=None) -> pd.DataFrame:
    """Fetch diaSources (direct detections) for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/sources", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_fp(diaObjectId, columns=None) -> pd.DataFrame:
    """Fetch forced photometry for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/fp", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


print("API helpers defined.")

## 2b. List available Fink Tags and Blocks (informational)

In [ ]:
df_api_tags = list_available_tags()
if not df_api_tags.empty:
    print(f"Fink Tags available from API ({len(df_api_tags)} entries):")
    display(df_api_tags)
else:
    print("No tags returned from API — using the hardcoded FINK_TAGS dictionary.")
    for k, v in FINK_TAGS.items():
        print(f"  {k:50s}  {v}")

In [ ]:
df_blocks = list_available_blocks()
if not df_blocks.empty:
    print(f"Fink Blocks ({len(df_blocks)} entries):")
    display(df_blocks)
else:
    print("No blocks returned (endpoint may not be available in this API version).")

## 3. Utility functions

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def angular_separation_arcsec(ra1, dec1, ra2_arr, dec2_arr):
    """
    Vectorised great-circle angular separation (arcsec) between a single
    reference point (ra1, dec1) and arrays of points (ra2_arr, dec2_arr).
    All angles in degrees. Uses the Vincenty formula for numerical stability.
    """
    ra1_r = np.radians(ra1)
    dec1_r = np.radians(dec1)
    ra2_r = np.radians(np.asarray(ra2_arr, dtype=float))
    dec2_r = np.radians(np.asarray(dec2_arr, dtype=float))
    dra = ra2_r - ra1_r
    num = np.sqrt(
        (np.cos(dec2_r) * np.sin(dra)) ** 2
        + (np.cos(dec1_r) * np.sin(dec2_r) - np.sin(dec1_r) * np.cos(dec2_r) * np.cos(dra)) ** 2
    )
    den = np.sin(dec1_r) * np.sin(dec2_r) + np.cos(dec1_r) * np.cos(dec2_r) * np.cos(dra)
    return np.degrees(np.arctan2(num, den)) * 3600.0  # deg → arcsec


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


def filter_lc(
    df_lc,
    mjd_col="r:midpointMjdTai",
    flux_col="r:psfFlux",
    ferr_col="r:psfFluxErr",
    band_col="r:band",
    snr_min=3.0,
):
    """
    Filter a raw light-curve DataFrame: drop low-SNR points, add magnitude columns.
    Works for both diaSources and forced photometry DataFrames.
    """
    df = df_lc.copy()
    for col in (mjd_col, flux_col, ferr_col, band_col):
        if col not in df.columns:
            return pd.DataFrame()
    df[flux_col] = pd.to_numeric(df[flux_col], errors="coerce")
    df[ferr_col] = pd.to_numeric(df[ferr_col], errors="coerce")
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")
    snr = df[flux_col].abs() / df[ferr_col].replace(0, np.nan)
    df = df[snr >= snr_min].sort_values(mjd_col).reset_index(drop=True)
    df = df.dropna(subset=[flux_col, ferr_col, mjd_col]).reset_index(drop=True)
    mag, mag_err = flux_to_mag(df[flux_col].values, df[ferr_col].values)
    df["mag"] = mag
    df["mag_err"] = mag_err
    df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)
    return df


print("Utility functions defined.")

## 4. Fetch tag alerts and filter by Deep Drilling Field

**Two-level loop**: for each tag, we fetch all alerts globally (one API call),  
then apply a **client-side spatial filter** for each DDF.  
This is efficient: `len(TAGS_TO_QUERY)` API calls total, not `len(TAGS) × len(DDFs)`.

Each candidate stores:
- `group` = tag name (replaces `classify_object()` from NB01)
- `field` = DDF name (e.g. `"COSMOS"`) — first DDF match if an object lies in several cones
- `tag`   = same as `group` (explicit copy for clarity)

If an object falls in multiple DDF cones, the **first DDF encountered** is recorded  
(priority follows `DEEP_FIELDS` insertion order). To record all DDFs, set  
`ALLOW_MULTI_FIELD = True` below — the object will appear multiple times  
with different `field` values.

In [ ]:
# If True: an object that falls in N DDF cones is kept N times (once per field).
# If False: only the first matching DDF is kept (original NB01 behaviour).
ALLOW_MULTI_FIELD = False

OID_COL = "r:diaObjectId"
NSRC_COL = "r:nDiaSources"
MJD_COL = "r:midpointMjdTai"
RA_COL = "r:ra"
DEC_COL = "r:dec"

all_candidates = {}  # key: oid (str)  or  (oid, field) if ALLOW_MULTI_FIELD
# value: {nDiaSources, group, tag, field, ra, dec}

# ── Per-tag global fetch ──────────────────────────────────────────────────────
tag_raw_cache = {}  # tag -> raw DataFrame (all alerts, before DDF filter)

for tag in TAGS_TO_QUERY:
    desc = FINK_TAGS.get(tag, "(no description)")
    print(f"\n── Tag: {tag!r}")
    print(f"   {desc}")
    print(f"   Fetching up to {N_PER_TAG} alerts globally ...")

    df_raw = fetch_tag_alerts(tag, n=N_PER_TAG, columns=CATALOG_COLUMNS)

    if df_raw.empty:
        print("   No alerts returned for this tag.")
        tag_raw_cache[tag] = pd.DataFrame()
        continue

    print(f"   Raw alerts returned : {len(df_raw)}")

    if OID_COL not in df_raw.columns:
        print(f"   WARNING: column {OID_COL!r} missing — skipping tag.")
        tag_raw_cache[tag] = pd.DataFrame()
        continue

    # Cast numeric columns
    for col in (NSRC_COL, MJD_COL, RA_COL, DEC_COL):
        if col in df_raw.columns:
            df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

    # Deduplicate: keep one row per diaObjectId (most recent alert)
    if MJD_COL in df_raw.columns:
        df_obj = (
            df_raw.sort_values(MJD_COL).drop_duplicates(subset=OID_COL, keep="last").reset_index(drop=True)
        )
    else:
        df_obj = df_raw.drop_duplicates(subset=OID_COL, keep="last").reset_index(drop=True)

    print(f"   Unique objects (global): {len(df_obj)}")
    tag_raw_cache[tag] = df_obj

    # ── Spatial filter: loop over DDFs ────────────────────────────────────────
    if RA_COL not in df_obj.columns or DEC_COL not in df_obj.columns:
        print(f"   WARNING: RA/Dec columns missing — cannot apply DDF spatial filter.")
        continue

    ra_arr = df_obj[RA_COL].values
    dec_arr = df_obj[DEC_COL].values

    for field_name, (ra_ddf, dec_ddf) in DEEP_FIELDS.items():
        sep = angular_separation_arcsec(ra_ddf, dec_ddf, ra_arr, dec_arr)
        in_cone = sep <= CONE_RADIUS
        df_field = df_obj[in_cone].copy()

        if df_field.empty:
            continue

        # Apply nDiaSources filter
        if NSRC_COL in df_field.columns:
            df_field = df_field[df_field[NSRC_COL].fillna(0) >= NP_MIN]

        if df_field.empty:
            continue

        print(
            f'   {field_name:15s}: {len(df_field)} objects in cone (sep <= {CONE_RADIUS:.0f}")'
            f", >= {NP_MIN} detections"
        )

        for _, row in df_field.iterrows():
            oid = str(row[OID_COL])
            nsrc = int(row.get(NSRC_COL, 0)) if NSRC_COL in row.index else 0

            # Key: (oid, field) if multi-field allowed, else oid
            key = (oid, field_name) if ALLOW_MULTI_FIELD else oid

            if key not in all_candidates:
                all_candidates[key] = {
                    "oid": oid,
                    "nDiaSources": nsrc,
                    "group": tag,  # tag name = group
                    "tag": tag,
                    "field": field_name,  # DDF name preserved here
                    "ra": float(row.get(RA_COL, float("nan"))),
                    "dec": float(row.get(DEC_COL, float("nan"))),
                }
            # If ALLOW_MULTI_FIELD=False and object already registered, skip
            # (first matching DDF wins, following NB01 convention)

    time.sleep(0.5)

print(f"\n=== Total unique candidates across all tags × DDFs: {len(all_candidates)} ===")

# ── Summary ───────────────────────────────────────────────────────────────────
tag_counts = collections.Counter(v["tag"] for v in all_candidates.values())
field_counts = collections.Counter(v["field"] for v in all_candidates.values())

print("\nDistribution by tag:")
for g, n in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {g:50s} {n:4d} objects")

print("\nDistribution by DDF:")
for f, n in sorted(field_counts.items(), key=lambda x: -x[1]):
    print(f"  {f:20s} {n:4d} objects")

## 5. Download light curves for all candidates

Same logic as notebook 01. The `field` (DDF name) is stored in `lc_cache`  
and propagated to all output files exactly like in NB01.

### Light-curve columns

**diaSources** (`src`): `r:diaObjectId`, `r:diaSourceId`, `r:midpointMjdTai`,  
`r:psfFlux`, `r:psfFluxErr`, `r:band`, `r:ra`, `r:dec`, `r:snr`,  
`r:visit`, `r:detector`, `r:x`, `r:y`, `r:xErr`, `r:yErr`

**Forced photometry** (`fp`): same minus `r:diaSourceId` and `r:snr`.

In [ ]:
COLS_SRC = (
    "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,r:band,r:ra,r:dec,r:snr,"
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr"
)

COLS_FP = (
    "r:diaObjectId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,r:visit,r:detector,r:x,r:y,r:xErr,r:yErr"
)

lc_cache = {}  # oid (str) → {src, fp, group (=tag), tag, field, nDiaSources}

MAX_OBJECTS = 200

# Sort by nDiaSources descending, cap at MAX_OBJECTS
# Note: if ALLOW_MULTI_FIELD=True, keys are (oid, field) tuples
cand_items = sorted(all_candidates.items(), key=lambda kv: kv[1]["nDiaSources"], reverse=True)[:MAX_OBJECTS]

print(f"Objects to fetch: {len(cand_items)}  (capped at {MAX_OBJECTS})")
if cand_items:
    print(f"nDiaSources range: {cand_items[0][1]['nDiaSources']} … {cand_items[-1][1]['nDiaSources']}")


def _cast_rubin_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Cast Rubin DRP integer columns to nullable Int64."""
    for col in ("r:visit", "r:detector"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in ("r:x", "r:y", "r:xErr", "r:yErr"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


for i, (key, meta) in enumerate(cand_items):
    oid = meta["oid"]
    tag = meta["tag"]
    field = meta["field"]

    # Avoid downloading the same oid twice when ALLOW_MULTI_FIELD=True
    # (light curve is the same regardless of DDF; field is stored per-candidate)
    if oid in lc_cache and not ALLOW_MULTI_FIELD:
        continue

    try:
        df_src = fetch_sources(oid, columns=COLS_SRC)
        df_fp = fetch_fp(oid, columns=COLS_FP)

        df_src = _cast_rubin_cols(df_src)
        df_fp = _cast_rubin_cols(df_fp)

        df_src_f = filter_lc(df_src)
        df_fp_f = filter_lc(df_fp)

        lc_cache[oid] = {
            "src": df_src_f,
            "fp": df_fp_f,
            "group": tag,  # tag name = group (replaces classify_object)
            "tag": tag,
            "field": field,  # DDF name — identical to NB01
            "nDiaSources": meta["nDiaSources"],
        }
        n_visits_src = df_src_f["r:visit"].dropna().nunique() if "r:visit" in df_src_f.columns else "?"
        n_visits_fp = df_fp_f["r:visit"].dropna().nunique() if "r:visit" in df_fp_f.columns else "N/A"
        print(
            f"  [{i + 1:3d}/{len(cand_items)}] {oid}  "
            f"src={len(df_src_f):4d} (visits={n_visits_src})  "
            f"fp={len(df_fp_f):4d} (visits={n_visits_fp})  "
            f"tag={tag}  field={field}"
        )
    except Exception as e:
        print(f"  [{i + 1:3d}/{len(cand_items)}] {oid}  ERROR: {e}")
    time.sleep(0.2)

print(f"\nDownloaded: {len(lc_cache)} objects")

## 6. Build per-tag (= per-group) object lists

In [ ]:
all_groups = sorted(set(d["group"] for d in lc_cache.values()))
group_oids = {g: [] for g in all_groups}
for oid, d in lc_cache.items():
    group_oids[d["group"]].append(oid)

print("Tag (= Group) → object count   [DDF breakdown]:")
for g in sorted(group_oids, key=lambda x: -len(group_oids[x])):
    desc = FINK_TAGS.get(g, "")
    fields_in_group = collections.Counter(lc_cache[o]["field"] for o in group_oids[g])
    field_str = "  ".join(f"{f}:{n}" for f, n in sorted(fields_in_group.items()))
    print(f"  {g:50s} {len(group_oids[g]):3d}   {field_str}")

# All tags are kept for downstream analysis
CALIB_GROUPS = list(all_groups)
print(f"\nAll {len(CALIB_GROUPS)} tag-groups will be analysed.")

## 7. Compute flatness metrics per tag

In [ ]:
flatness_rows = []

for oid, data in lc_cache.items():
    group = data["group"]
    field = data["field"]
    frames = [df for df in (data["fp"], data["src"]) if not df.empty]
    if not frames:
        continue
    df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)
    for band in BANDS:
        df_b = df_all[df_all["r:band"] == band]
        if len(df_b) < 5:
            continue
        rms = rms_variability(df_b["r:psfFlux"].values)
        flatness_rows.append(
            {
                "group": group,
                "tag": group,
                "field": field,
                "diaObjectId": oid,
                "band": band,
                "n_pts": len(df_b),
                "rms_var": rms,
                "mean_flux_nJy": float(df_b["r:psfFlux"].mean()),
            }
        )

df_flat = pd.DataFrame(flatness_rows)

if not df_flat.empty:
    print(f"Flatness table: {len(df_flat)} rows")
    print("\nMedian flatness by tag:")
    print(df_flat.groupby("group")[["n_pts", "rms_var"]].median().round(4))
    print("\nMedian flatness by DDF:")
    print(df_flat.groupby("field")[["n_pts", "rms_var"]].median().round(4))
else:
    print("No flatness data — check light curve downloads.")

df_flat.to_csv(os.path.join(DIR_DATA, "flatness_metrics.csv"), index=False)
print("\nSaved flatness_metrics.csv")

## 7b. Visit summary — diaSources and Forced Photometry

The `field` (DDF name) column is included in both summary tables,  
identical to notebook 01.

In [ ]:
def build_visit_summary(lc_cache: dict, source_type: str = "src") -> pd.DataFrame:
    """
    Build a visit-level summary DataFrame from the light curve cache.
    One row per (diaObjectId, visit, detector, band).
    Includes 'field' (DDF name) and 'tag' columns.
    """
    rows = []
    for oid, data in lc_cache.items():
        df = data.get(source_type, pd.DataFrame())
        group = data["group"]
        field = data.get("field", "unknown")
        if df.empty:
            continue
        if "r:visit" not in df.columns:
            continue

        group_cols = ["r:visit", "r:band"]
        if "r:detector" in df.columns:
            group_cols.insert(1, "r:detector")

        for keys, grp in df.groupby(group_cols, dropna=False):
            if not isinstance(keys, tuple):
                keys = (keys,)
            key_dict = dict(zip(group_cols, keys))

            row = {
                "diaObjectId": oid,
                "group": group,
                "tag": group,
                "field": field,  # DDF name — same as NB01
                "source_type": source_type,
                "visit": key_dict.get("r:visit"),
                "detector": key_dict.get("r:detector"),
                "band": key_dict.get("r:band"),
                "n_det": len(grp),
            }

            if "r:midpointMjdTai" in grp.columns:
                mjd_vals = pd.to_numeric(grp["r:midpointMjdTai"], errors="coerce")
                row["mjd_first"] = float(mjd_vals.min()) if mjd_vals.notna().any() else np.nan
            else:
                row["mjd_first"] = np.nan

            for col_in, col_out in [
                ("r:x", "x_mean"),
                ("r:y", "y_mean"),
                ("r:xErr", "xErr_mean"),
                ("r:yErr", "yErr_mean"),
            ]:
                if col_in in grp.columns:
                    vals = pd.to_numeric(grp[col_in], errors="coerce")
                    row[col_out] = float(vals.mean()) if vals.notna().any() else np.nan
                else:
                    row[col_out] = np.nan

            if "r:psfFlux" in grp.columns:
                flux = pd.to_numeric(grp["r:psfFlux"], errors="coerce")
                row["psfFlux_mean"] = float(flux.mean()) if flux.notna().any() else np.nan
            else:
                row["psfFlux_mean"] = np.nan

            rows.append(row)

    if not rows:
        return pd.DataFrame()

    df_out = pd.DataFrame(rows)
    for col in ("visit", "detector"):
        if col in df_out.columns:
            df_out[col] = pd.to_numeric(df_out[col], errors="coerce").astype("Int64")
    df_out = df_out.sort_values(["diaObjectId", "visit", "detector", "band"]).reset_index(drop=True)
    return df_out


df_visit_summary_src = build_visit_summary(lc_cache, source_type="src")
df_visit_summary_fp = build_visit_summary(lc_cache, source_type="fp")

print("=== diaSources visit summary ===")
if not df_visit_summary_src.empty:
    print(f"  Rows             : {len(df_visit_summary_src)}")
    print(f"  Unique objects   : {df_visit_summary_src['diaObjectId'].nunique()}")
    print(f"  Unique visits    : {df_visit_summary_src['visit'].dropna().nunique()}")
    print(f"  Unique DDFs      : {sorted(df_visit_summary_src['field'].dropna().unique())}")
    print(f"  Unique tags      : {sorted(df_visit_summary_src['tag'].dropna().unique())}")
    display(df_visit_summary_src.head(5))
else:
    print("  No diaSources visit data available.")

print("\n=== Forced Photometry visit summary ===")
if not df_visit_summary_fp.empty:
    print(f"  Rows             : {len(df_visit_summary_fp)}")
    print(f"  Unique objects   : {df_visit_summary_fp['diaObjectId'].nunique()}")
    print(f"  Unique visits    : {df_visit_summary_fp['visit'].dropna().nunique()}")
    display(df_visit_summary_fp.head(5))
else:
    print("  r:visit not available in forced photometry (expected for some API versions).")

if not df_visit_summary_src.empty:
    path_src = os.path.join(DIR_DATA, "visit_summary_src.csv")
    df_visit_summary_src.to_csv(path_src, index=False)
    print(f"\nSaved: {path_src}")

if not df_visit_summary_fp.empty:
    path_fp = os.path.join(DIR_DATA, "visit_summary_fp.csv")
    df_visit_summary_fp.to_csv(path_fp, index=False)
    print(f"Saved: {path_fp}")

## 8. Plot: flatness distribution per tag

In [ ]:
if df_flat.empty:
    print("No data.")
else:
    groups_present = [g for g in all_groups if g in df_flat["group"].unique()]
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["rms_var"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by Fink tag (all DDFs combined)", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01_flatness_boxplot_by_tag")
    plt.show()

In [ ]:
# ── Flatness per DDF (all tags combined) ──────────────────────────────────────
if not df_flat.empty:
    fields_present = sorted(df_flat["field"].dropna().unique())
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = fields_present

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_field = [df_b[df_b["field"] == f]["rms_var"].dropna().values for f in fields_present]
        bp = ax.boxplot(data_per_field, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by DDF (all tags combined)", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01b_flatness_boxplot_by_field")
    plt.show()

## 9. Plot light curves — top-NC objects per tag

The DDF name (`field`) is displayed in the y-axis label of each light curve row,  
just as in notebook 01.

In [ ]:
def rank_oids(oid_list, nc=NC):
    scored = [(o, len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"])) for o in oid_list if o in lc_cache]
    return [o for o, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def plot_lc_grid(oid_list, group, mode="flux", nc=NC):
    top = rank_oids(oid_list, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for tag {group}.")
        return
    fig, axes = plt.subplots(
        n_obj, len(BANDS), figsize=(2.8 * len(BANDS), 2.6 * n_obj), sharex=False, sharey=False, squeeze=False
    )

    for row_idx, oid in enumerate(top):
        d = lc_cache[oid]
        field = d.get("field", "")
        frames = [df for df in (d.get("fp", pd.DataFrame()), d.get("src", pd.DataFrame())) if not df.empty]
        if not frames:
            continue
        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        ax_first_band = None
        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = df_all[df_all["r:band"] == band].sort_values("r:midpointMjdTai")
            if len(df_b) < 3:
                ax.set_visible(False)
                continue
            if ax_first_band is None:
                ax_first_band = ax

            if mode == "flux":
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
            else:
                mask = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
            df_b = df_b[mask].reset_index(drop=True)
            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            dt = df_b["r:midpointMjdTai"].values
            dt = dt - dt.min()
            color = BAND_COLORS.get(band, "gray")

            if mode == "flux":
                y, yerr = df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values
            else:
                y, yerr = df_b["mag"].values, df_b["mag_err"].values
                ax.invert_yaxis()

            ax.errorbar(dt, y, yerr=yerr, fmt="o", ms=3, lw=0.8, elinewidth=0.8, color=color, alpha=0.8)
            rms = rms_variability(df_b["r:psfFlux"].values)
            ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}", fontsize=7, pad=2, color=color)
            ax.set_xlabel("Δt (days)", fontsize=7)
            ax.tick_params(labelsize=7)

        label_ax = ax_first_band if ax_first_band is not None else axes[row_idx][0]
        # Include DDF name in y-axis label (same as NB01)
        field_label = f"  [{field}]" if field else ""
        label_ax.set_ylabel(f"{oid}{field_label}\n{'flux (nJy)' if mode == 'flux' else 'AB mag'}", fontsize=9)

    fig.suptitle(f"Tag: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("Plot functions defined.")

In [ ]:
groups_to_plot = [g for g in all_groups if len(group_oids.get(g, [])) >= 2]

for group in groups_to_plot:
    print(f"\n=== {group} ({len(group_oids[group])} objects) ===")
    print(f"    {FINK_TAGS.get(group, '')}")
    plot_lc_grid(group_oids[group], group, mode="flux")

In [ ]:
for group in groups_to_plot:
    print(f"\n=== {group} (magnitude) ===")
    plot_lc_grid(group_oids[group], group, mode="mag")

## 10. Summary scatter plot

In [ ]:
if df_flat.empty:
    print("No data.")
else:
    summary = (
        df_flat.groupby(["group", "band"])
        .agg(
            median_rms=("rms_var", "median"),
            mean_flux=("mean_flux_nJy", "mean"),
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
        )
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary["band"].unique()]
    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5 * len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary["band"] == band]
        ax.scatter(
            df_b["mean_flux"],
            df_b["median_rms"],
            s=80 * np.sqrt(df_b["n_pts"] / max(df_b["n_pts"].max(), 1) + 0.1),
            c=BAND_COLORS.get(band, "gray"),
            alpha=0.8,
            edgecolors="k",
            linewidths=0.5,
        )
        for _, row in df_b.iterrows():
            ax.annotate(
                row["group"],
                (row["mean_flux"], row["median_rms"]),
                fontsize=6,
                ha="left",
                va="bottom",
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Mean flux (nJy)")
        ax.set_ylabel("Median σ/<f>")
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.axhline(0.05, ls="--", color="green", lw=1, alpha=0.6, label="5%")
        ax.legend(fontsize=7)

    fig.suptitle("Fink tag variability summary (objects in DDFs)", fontsize=11, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("03_tag_summary_scatter")
    plt.show()

    print("\nFull summary table:")
    display(summary.sort_values(["band", "median_rms"]).head(40))

## 11. Save all light curves to Parquet

Each parquet file contains a `field` column (DDF name) and a `tag` column,  
identical to notebook 01 which stored `group` and `field`.

In [ ]:
for group in all_groups:
    oids = group_oids[group]
    all_fp, all_src = [], []
    for oid in oids:
        d = lc_cache.get(oid, {})
        for df, store in ((d.get("fp", pd.DataFrame()), all_fp), (d.get("src", pd.DataFrame()), all_src)):
            if not df.empty:
                tmp = df.copy()
                tmp["diaObjectId"] = oid
                tmp["group"] = group
                tmp["tag"] = group
                tmp["field"] = d.get("field", "unknown")  # DDF name
                store.append(tmp)
    safe = group.replace("/", "_")
    for store, tag_suffix in ((all_fp, "fp"), (all_src, "src")):
        if store:
            path = os.path.join(DIR_DATA, f"{safe}_{tag_suffix}.parquet")
            pd.concat(store, ignore_index=True).to_parquet(path, index=False)
            print(f"  Saved {path}")

print("\nAll data saved.")

## 12. Build and save the global visit index

In [ ]:
if not df_visit_summary_src.empty:
    idx_path = os.path.join(DIR_DATA, "visit_index.csv")
    df_visit_summary_src.to_csv(idx_path, index=False)
    print(
        f"visit_index.csv saved ({len(df_visit_summary_src)} rows, "
        f"{df_visit_summary_src['visit'].dropna().nunique()} unique visits, "
        f"{df_visit_summary_src['field'].dropna().nunique()} DDFs)"
    )

if not df_visit_summary_fp.empty:
    idx_fp_path = os.path.join(DIR_DATA, "visit_index_fp.csv")
    df_visit_summary_fp.to_csv(idx_fp_path, index=False)
    print(
        f"visit_index_fp.csv saved ({len(df_visit_summary_fp)} rows, "
        f"{df_visit_summary_fp['visit'].dropna().nunique()} unique visits)"
    )

## 13. Final ranking — variability by tag

In [ ]:
if df_flat.empty:
    print("No flatness data.")
else:
    ranking = (
        df_flat.groupby("group")
        .agg(
            n_objects=("diaObjectId", "nunique"),
            n_points=("n_pts", "sum"),
            median_rms=("rms_var", "median"),
            mean_flux_nJy=("mean_flux_nJy", "mean"),
        )
        .sort_values("median_rms")
        .reset_index()
    )
    ranking["mean_mag"] = -2.5 * np.log10(ranking["mean_flux_nJy"] / AB_FLUX_ZERO)
    ranking["description"] = ranking["group"].map(FINK_TAGS).fillna("")

    print("Ranking by photometric variability (ascending RMS) — all Fink tags in DDFs:")
    print("=" * 110)
    print(
        ranking[["group", "n_objects", "n_points", "median_rms", "mean_mag", "description"]].to_string(
            index=False, float_format="{:.4f}".format
        )
    )

    ranking.to_csv(os.path.join(DIR_DATA, "tag_ranking.csv"), index=False)
    print("\nSaved tag_ranking.csv")

    # ── Breakdown by (tag, DDF) ───────────────────────────────────────────────
    print("\nMedian RMS by (tag, DDF):")
    breakdown = (
        df_flat.groupby(["group", "field"])
        .agg(n_obj=("diaObjectId", "nunique"), median_rms=("rms_var", "median"))
        .reset_index()
        .sort_values(["group", "median_rms"])
    )
    display(breakdown)